## 2.1 理论计算题（马尔可夫模型）

已知字符序列 "ababc"，采用一阶马尔可夫模型，使用拉普拉斯平滑（加1平滑），词汇表 $V = \{a, b, c\}$。

### 统计转移计数

序列 "a b a b c" 中的转移对：
- $a \to b$：出现 2 次（位置 0→1, 2→3）
- $b \to a$：出现 1 次（位置 1→2）
- $b \to c$：出现 1 次（位置 3→4）
- 其余转移均未出现

### 拉普拉斯平滑

对每个当前字符 $x$，其所有可能的下一字符 $y \in V$ 的计数加 1：

| 当前字符 | $\to a$ (平滑后) | $\to b$ (平滑后) | $\to c$ (平滑后) | 总计 |
|:---:|:---:|:---:|:---:|:---:|
| a | 0+1=1 | 2+1=3 | 0+1=1 | 5 |
| b | 1+1=2 | 0+1=1 | 1+1=2 | 5 |
| c | 0+1=1 | 0+1=1 | 0+1=1 | 3 |

### 1. $p(a \mid b)$

$$p(a \mid b) = \frac{\text{count}(b \to a)}{\sum_{y \in V} \text{count}(b \to y)} = \frac{2}{5} = 0.4$$

### 2. $p(c \mid b)$

$$p(c \mid b) = \frac{\text{count}(b \to c)}{\sum_{y \in V} \text{count}(b \to y)} = \frac{2}{5} = 0.4$$

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表并生成 n-gram 特征序列和标签。

    参数:
        text: 输入文本字符串
        n: 滑动窗口大小（上下文词数）

    返回:
        vocab: 词汇表字典 {词: id}
        features: n-gram 特征列表
        labels: 下一个词标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. 按空格分词
    words = text.split()

    if len(words) == 0:
        return {}, [], []

    # 3. 构建词汇表（按出现频率排序，从0开始分配ID）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}

    # 4. 用滑动窗口生成 n-gram 特征和下一个词标签
    features = []
    labels = []

    for i in range(len(words) - n):
        feature = words[i:i+n]     # 连续的 n 个词
        label = words[i+n]         # 紧接着的第 n+1 个词
        features.append(feature)
        labels.append(label)

    return vocab, features, labels


# 测试示例
text = "The time machine"
vocab, features, labels = preprocess_text(text, 2)
print("词汇表:", vocab)
print("特征序列:", features)
print("标签序列:", labels)

# 额外测试：较长文本
text2 = "The time machine is a science fiction novella by H G Wells"
vocab2, features2, labels2 = preprocess_text(text2, 3)
print("\n词汇表2:", vocab2)
print("特征序列2:", features2)
print("标签序列2:", labels2)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
标签序列: ['machine']

词汇表2: {'a': 0, 'by': 1, 'fiction': 2, 'g': 3, 'h': 4, 'is': 5, 'machine': 6, 'novella': 7, 'science': 8, 'the': 9, 'time': 10, 'wells': 11}
特征序列2: [['the', 'time', 'machine'], ['time', 'machine', 'is'], ['machine', 'is', 'a'], ['is', 'a', 'science'], ['a', 'science', 'fiction'], ['science', 'fiction', 'novella'], ['fiction', 'novella', 'by'], ['novella', 'by', 'h'], ['by', 'h', 'g']]
标签序列2: ['is', 'a', 'science', 'fiction', 'novella', 'by', 'h', 'g', 'wells']


## 3.1 理论计算题（线性 RNN 的 BPTT 梯度推导）

### 模型定义

线性 RNN（无偏置）：
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$
$$o_t = W_{oh} h_t$$

平方损失函数：
$$L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$$

### 推导 $\frac{\partial L}{\partial W_{hh}}$

定义误差信号 $\delta_t = \frac{\partial L}{\partial h_t}$，则：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}$$

通过链式法则展开 $\frac{\partial h_t}{\partial W_{hh}}$：
$$\frac{\partial h_t}{\partial W_{hh}} = h_{t-1}^\top + W_{hh} \frac{\partial h_{t-1}}{\partial W_{hh}}$$

递归展开：
$$\frac{\partial h_t}{\partial W_{hh}} = \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} W_{hh} \right) h_{k-1}^\top$$

其中 $\delta_t = W_{oh}^\top (o_t - y_t)$（对于线性输出层）。

因此：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} W_{oh}^\top (o_t - y_t) \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} W_{hh} \right) h_{k-1}^\top$$

### 梯度消失/爆炸条件

梯度中包含 $W_{hh}$ 的幂次项 $\prod_{j=k+1}^{t} W_{hh}$，其特征值决定了梯度的行为：

- **梯度爆炸**：若 $W_{hh}$ 的最大特征值 $|\lambda_{\max}| > 1$，则连乘项随 $t-k$ 增大而指数增长
- **梯度消失**：若 $W_{hh}$ 的最大特征值 $|\lambda_{\max}| < 1$，则连乘项随 $t-k$ 增大而指数衰减至 0
- **稳定传播**：仅当 $|\lambda_{\max}| \approx 1$ 时梯度才能有效传播

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播（tanh 激活）

    参数:
        x_t: 输入 (batch_size, input_size)
        h_prev: 上一隐藏状态 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重 (hidden_size, hidden_size)
        b_h: 隐藏层偏置 (hidden_size,)

    返回:
        h_t: 当前隐藏状态 (batch_size, hidden_size)
        cache: 反向传播所需的中间值
    """
    # 线性变换
    z = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    # tanh 激活
    h_t = np.tanh(z)
    cache = (x_t, h_prev, h_t, z, W_hx, W_hh)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN 单元单步反向传播

    参数:
        dh_next: 上游梯度，损失对 h_t 的梯度 (batch_size, hidden_size)
        cache: 前向传播保存的中间值

    返回:
        dx_t: 输入梯度 (batch_size, input_size)
        dh_prev: 上一隐藏状态梯度 (batch_size, hidden_size)
        dW_hx: W_hx 梯度 (input_size, hidden_size)
        dW_hh: W_hh 梯度 (hidden_size, hidden_size)
        db_h: b_h 梯度 (hidden_size,)
    """
    x_t, h_prev, h_t, z, W_hx, W_hh = cache

    # tanh 反向: d(tanh(z))/dz = 1 - tanh(z)^2 = 1 - h_t^2
    dz = dh_next * (1 - h_t ** 2)

    # 各参数梯度
    dx_t = np.dot(dz, W_hx.T)
    dh_prev = np.dot(dz, W_hh.T)
    dW_hx = np.dot(x_t.T, dz)
    dW_hh = np.dot(h_prev.T, dz)
    db_h = np.sum(dz, axis=0)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试
np.random.seed(42)
batch_size, input_size, hidden_size = 3, 4, 5
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)

# 前向
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
print("h_t shape:", h_t.shape)
print("h_t:\n", h_t)

# 反向（使用随机上游梯度进行测试）
dh_next = np.random.randn(batch_size, hidden_size) * 0.1
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print("\ndx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)

# 数值梯度检查
epsilon = 1e-5
W_hx_plus = W_hx.copy()
W_hx_plus[0, 0] += epsilon
h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hx_plus, W_hh, b_h)
loss_plus = np.sum(h_t_plus)

W_hx_minus = W_hx.copy()
W_hx_minus[0, 0] -= epsilon
h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hx_minus, W_hh, b_h)
loss_minus = np.sum(h_t_minus)

numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
dh_for_check = np.ones_like(h_t)
_, _, dW_hx_check, _, _ = rnn_cell_backward(dh_for_check, cache)
print(f"\n数值梯度 W_hx[0,0]: {numerical_grad:.6f}")
print(f"解析梯度 W_hx[0,0]: {dW_hx_check[0, 0]:.6f}")

h_t shape: (3, 5)
h_t:
 [[ 0.04327787 -0.27504859 -0.55586809 -0.26860544  0.06152714]
 [-0.3568583  -0.43742518 -0.24299154  0.28025916 -0.01150663]
 [ 0.07035202 -0.11091096 -0.0105794  -0.16803976 -0.1123908 ]]

dx_t shape: (3, 4)
dh_prev shape: (3, 5)
dW_hx shape: (4, 5)
dW_hh shape: (5, 5)
db_h shape: (5,)

数值梯度 W_hx[0,0]: -0.175701
解析梯度 W_hx[0,0]: -0.175701


## 4.1 理论计算题（深度双向 RNN 参数计算）

### 模型结构

- $L$ 层双向 RNN
- 每层隐藏单元数 $H$
- 输入维度 $D$，输出维度 $O$

标准 RNN 单元参数（每个方向）：
$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

### 逐层参数分析

#### 第 1 层（输入来自原始数据，维度 $D$）

- **前向**：$W_{ih}^{f,1} \in \mathbb{R}^{H \times D}$，$b_{ih}^{f,1} \in \mathbb{R}^{H}$，$W_{hh}^{f,1} \in \mathbb{R}^{H \times H}$，$b_{hh}^{f,1} \in \mathbb{R}^{H}$
- **后向**：$W_{ih}^{b,1} \in \mathbb{R}^{H \times D}$，$b_{ih}^{b,1} \in \mathbb{R}^{H}$，$W_{hh}^{b,1} \in \mathbb{R}^{H \times H}$，$b_{hh}^{b,1} \in \mathbb{R}^{H}$

第 1 层参数量：$2 \times (HD + H + H^2 + H) = 2H^2 + 2HD + 4H$

#### 第 $l$ 层（$l = 2, \ldots, L$，输入为上层拼接的前向+后向隐藏状态，维度 $2H$）

- **前向**：$W_{ih}^{f,l} \in \mathbb{R}^{H \times 2H}$，$b_{ih}^{f,l} \in \mathbb{R}^{H}$，$W_{hh}^{f,l} \in \mathbb{R}^{H \times H}$，$b_{hh}^{f,l} \in \mathbb{R}^{H}$
- **后向**：$W_{ih}^{b,l} \in \mathbb{R}^{H \times 2H}$，$b_{ih}^{b,l} \in \mathbb{R}^{H}$，$W_{hh}^{b,l} \in \mathbb{R}^{H \times H}$，$b_{hh}^{b,l} \in \mathbb{R}^{H}$

每层（$l \ge 2$）参数量：$2 \times (H \cdot 2H + H + H^2 + H) = 2(3H^2 + 2H) = 6H^2 + 4H$

#### 输出层

最后一层双向 RNN 的拼接隐藏状态（维度 $2H$）映射到输出维度 $O$：
- $W_{out} \in \mathbb{R}^{O \times 2H}$，$b_{out} \in \mathbb{R}^{O}$
- 参数量：$2HO + O$

### 总参数表达式

$$\begin{aligned}
N_{total} &= (2H^2 + 2HD + 4H) + (L-1)(6H^2 + 4H) + (2HO + O) \\
&= 2H^2 + 2HD + 4H + 6H^2L - 6H^2 + 4HL - 4H + 2HO + O \\
&= 6H^2L - 4H^2 + 2HD + 4HL + 2HO + O \\
&= H(6HL - 4H + 2D + 4L + 2O) + O
\end{aligned}$$

In [3]:
import numpy as np

def bidirectional_rnn_encoder(X, hidden_dim):
    """
    双向 RNN 编码器（NumPy 手动实现，单层）

    参数:
        X: 输入序列 (seq_len, batch, input_dim)
        hidden_dim: 隐藏层维度

    返回:
        all_hidden: 每个时间步拼接的前向+后向隐藏状态
                    (seq_len, batch, 2 * hidden_dim)
        final_hidden: 最终时间步拼接的隐藏状态 (batch, 2 * hidden_dim)
    """
    seq_len, batch, input_dim = X.shape

    # 随机初始化参数
    np.random.seed(42)

    # 前向 RNN 参数
    W_ih_f = np.random.randn(hidden_dim, input_dim) * 0.1
    W_hh_f = np.random.randn(hidden_dim, hidden_dim) * 0.1
    b_ih_f = np.zeros(hidden_dim)
    b_hh_f = np.zeros(hidden_dim)

    # 后向 RNN 参数
    W_ih_b = np.random.randn(hidden_dim, input_dim) * 0.1
    W_hh_b = np.random.randn(hidden_dim, hidden_dim) * 0.1
    b_ih_b = np.zeros(hidden_dim)
    b_hh_b = np.zeros(hidden_dim)

    # 前向 RNN（从 t=0 到 t=seq_len-1）
    h_forward = np.zeros((batch, hidden_dim))
    forward_states = []
    for t in range(seq_len):
        x_t = X[t]  # (batch, input_dim)
        h_forward = np.tanh(
            x_t @ W_ih_f.T + b_ih_f + h_forward @ W_hh_f.T + b_hh_f
        )
        forward_states.append(h_forward)

    # 后向 RNN（从 t=seq_len-1 到 t=0）
    h_backward = np.zeros((batch, hidden_dim))
    backward_states = [None] * seq_len
    for t in range(seq_len - 1, -1, -1):
        x_t = X[t]
        h_backward = np.tanh(
            x_t @ W_ih_b.T + b_ih_b + h_backward @ W_hh_b.T + b_hh_b
        )
        backward_states[t] = h_backward

    # 拼接每个时间步的前向和后向隐藏状态
    all_hidden = []
    for t in range(seq_len):
        concat = np.concatenate([forward_states[t], backward_states[t]], axis=-1)
        all_hidden.append(concat)
    all_hidden = np.stack(all_hidden, axis=0)  # (seq_len, batch, 2*hidden_dim)

    # 最终时间步的拼接隐藏状态
    final_hidden = np.concatenate([forward_states[-1], backward_states[0]], axis=-1)
    # (batch, 2*hidden_dim)

    return all_hidden, final_hidden


# 测试
np.random.seed(42)
seq_len, batch, input_dim, hidden_dim = 5, 3, 4, 6

X = np.random.randn(seq_len, batch, input_dim)
all_hidden, final_hidden = bidirectional_rnn_encoder(X, hidden_dim)

print("输入 X shape:", X.shape)
print("每步拼接隐藏状态 shape:", all_hidden.shape)
print("最终拼接隐藏状态 shape:", final_hidden.shape)
print("\n最终隐藏状态:\n", final_hidden)
print("\n验证 - 第一个时间步的拼接隐藏状态:\n", all_hidden[0])

输入 X shape: (5, 3, 4)
每步拼接隐藏状态 shape: (5, 3, 12)
最终拼接隐藏状态 shape: (3, 12)

最终隐藏状态:
 [[ 0.02387024 -0.02446598 -0.05886213  0.27428817 -0.05823098  0.11264999
  -0.29598122  0.1909149   0.27703425 -0.04059316 -0.35345589 -0.03622493]
 [ 0.16607361  0.21000259 -0.01625783 -0.32910758 -0.12141584 -0.22424261
  -0.29530822  0.02708907  0.16616833 -0.33644143 -0.10004939  0.17547422]
 [ 0.14008421  0.18227342 -0.04695059 -0.12214338 -0.12122773 -0.27313098
   0.08555718 -0.02936713 -0.18714618  0.23168935  0.13543206  0.04458524]]

验证 - 第一个时间步的拼接隐藏状态:
 [[ 0.29176667  0.20770622 -0.13101083 -0.15756291 -0.31722936 -0.13585202
  -0.29598122  0.1909149   0.27703425 -0.04059316 -0.35345589 -0.03622493]
 [ 0.20770622  0.30883016 -0.11018647 -0.26959015 -0.23116826 -0.1270186
  -0.29530822  0.02708907  0.16616833 -0.33644143 -0.10004939  0.17547422]
 [-0.13101083 -0.11018647  0.0943622  -0.00904306  0.17076473 -0.0178309
   0.08555718 -0.02936713 -0.18714618  0.23168935  0.13543206  0.04458524]]


## 5.1 理论计算题（Skip-gram 负采样）

### 问题设定

- 中心词 $w_c$，上下文词 $w_o$
- 词向量：中心词向量 $v_c$，上下文词向量 $u_o$
- 负采样 $K$ 个负样本，负样本词向量 $u_{n_k}$（$k = 1, \ldots, K$）

### 目标函数（最大化对数似然）

对于每个 $(w_c, w_o)$ 正样本对：
$$\mathcal{L} = \log \sigma(u_o^\top v_c) + \sum_{k=1}^{K} \mathbb{E}_{w_k \sim P_n(w)} \left[ \log \sigma(-u_{n_k}^\top v_c) \right]$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 为 sigmoid 函数。

损失函数（最小化负对数似然）：
$$J = -\log \sigma(u_o^\top v_c) - \sum_{k=1}^{K} \log \sigma(-u_{n_k}^\top v_c)$$

### 负采样策略

负样本从噪声分布 $P_n(w)$ 中采样，通常取**一元分布（unigram distribution）的 $3/4$ 次方**：

$$P_n(w) = \frac{U(w)^{3/4}}{\sum_{w'} U(w')^{3/4}}$$

其中 $U(w)$ 为词 $w$ 在语料中的出现频率。$3/4$ 次方的作用是**平滑**分布，适当提高低频词被采样的概率，降低高频词被采样的概率。

### 梯度

对正样本词向量 $u_o$ 的梯度：
$$\frac{\partial J}{\partial u_o} = -(1 - \sigma(u_o^\top v_c)) \cdot v_c = (\sigma(u_o^\top v_c) - 1) \cdot v_c$$

对负样本词向量 $u_{n_k}$ 的梯度：
$$\frac{\partial J}{\partial u_{n_k}} = \sigma(u_{n_k}^\top v_c) \cdot v_c$$

对中心词向量 $v_c$ 的梯度：
$$\frac{\partial J}{\partial v_c} = (\sigma(u_o^\top v_c) - 1) \cdot u_o + \sum_{k=1}^{K} \sigma(u_{n_k}^\top v_c) \cdot u_{n_k}$$

In [4]:
import numpy as np

def softmax(z):
    """Softmax 函数（数值稳定版本）"""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def cross_entropy_loss(probs, target_indices):
    """交叉熵损失"""
    batch_size = probs.shape[0]
    target_probs = probs[np.arange(batch_size), target_indices]
    loss = -np.mean(np.log(target_probs + 1e-8))
    return loss


def cbow_forward(context_indices, W, W_out):
    """
    CBOW 模型前向传播

    参数:
        context_indices: 上下文词索引 (batch_size, context_size)
        W: 输入词嵌入矩阵 (V, d)
        W_out: 输出权重矩阵 (d, V)

    返回:
        hidden: 隐藏层向量 (batch_size, d)
        probs: 输出概率分布 (batch_size, V)
    """
    batch_size, context_size = context_indices.shape

    # 1. 查找上下文词嵌入 (batch_size, context_size, d)
    context_embeds = W[context_indices]

    # 2. 平均上下文向量作为隐藏层 (batch_size, d)
    hidden = np.mean(context_embeds, axis=1)

    # 3. 输出 logits (batch_size, V)
    logits = np.dot(hidden, W_out)

    # 4. Softmax 概率分布
    probs = softmax(logits)

    return hidden, probs


def cbow_loss(context_indices, center_indices, W, W_out):
    """
    CBOW 模型完整前向 + 损失计算

    参数:
        context_indices: 上下文词索引 (batch_size, context_size)
        center_indices: 中心词索引/目标 (batch_size,)
        W: 输入词嵌入矩阵 (V, d)
        W_out: 输出权重矩阵 (d, V)

    返回:
        loss: 交叉熵损失值
    """
    _, probs = cbow_forward(context_indices, W, W_out)
    loss = cross_entropy_loss(probs, center_indices)
    return loss


# 测试
np.random.seed(42)
V, d, context_size, batch_size = 10, 8, 4, 3

# 随机初始化参数
W = np.random.randn(V, d) * 0.1
W_out = np.random.randn(d, V) * 0.1

# 随机上下文词索引和中心词索引
context_indices = np.random.randint(0, V, (batch_size, context_size))
center_indices = np.random.randint(0, V, batch_size)

print("词汇表大小 V:", V)
print("嵌入维度 d:", d)
print("上下文窗口大小 context_size:", context_size)
print("批次大小 batch_size:", batch_size)
print("\n上下文词索引:\n", context_indices)
print("中心词索引:", center_indices)

loss = cbow_loss(context_indices, center_indices, W, W_out)
print(f"\nCBOW 交叉熵损失: {loss:.4f}")

# 验证输出概率和为1
hidden, probs = cbow_forward(context_indices, W, W_out)
print(f"概率和: {probs.sum(axis=1)}")
print(f"隐藏层 shape: {hidden.shape}")
print(f"概率分布 shape: {probs.shape}")

词汇表大小 V: 10
嵌入维度 d: 8
上下文窗口大小 context_size: 4
批次大小 batch_size: 3

上下文词索引:
 [[0 3 0 4]
 [3 7 7 6]
 [2 0 0 2]]
中心词索引: [5 6 5]

CBOW 交叉熵损失: 2.3005
概率和: [1. 1. 1.]
隐藏层 shape: (3, 8)
概率分布 shape: (3, 10)


## 6.1 理论计算题（缩放点积注意力数值计算）

### 给定矩阵

$$Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix} \in \mathbb{R}^{2 \times 4}$$

$$K = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 1 & 0 & 0 \end{bmatrix} \in \mathbb{R}^{3 \times 4}$$

$$V = \begin{bmatrix} 1 & 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 & 0 \end{bmatrix} \in \mathbb{R}^{3 \times 5}$$

$d_k = 4$，缩放因子 $\sqrt{d_k} = 2$

### 步骤 1：计算得分矩阵 $S = \frac{Q K^\top}{\sqrt{d_k}}$

$$Q K^\top = \begin{bmatrix}
1 \cdot 1 + 0 \cdot 0 + 1 \cdot 1 + 0 \cdot 0 & 1 \cdot 0 + 0 \cdot 1 + 1 \cdot 0 + 0 \cdot 1 & 1 \cdot 1 + 0 \cdot 1 + 1 \cdot 0 + 0 \cdot 0 \\
0 \cdot 1 + 1 \cdot 0 + 0 \cdot 1 + 1 \cdot 0 & 0 \cdot 0 + 1 \cdot 1 + 0 \cdot 0 + 1 \cdot 1 & 0 \cdot 1 + 1 \cdot 1 + 0 \cdot 0 + 1 \cdot 0
\end{bmatrix}
= \begin{bmatrix} 2 & 0 & 1 \\ 0 & 2 & 1 \end{bmatrix}$$

$$S = \frac{Q K^\top}{\sqrt{4}} = \frac{1}{2} \begin{bmatrix} 2 & 0 & 1 \\ 0 & 2 & 1 \end{bmatrix} = \begin{bmatrix} 1.0 & 0.0 & 0.5 \\ 0.0 & 1.0 & 0.5 \end{bmatrix}$$

### 步骤 2：Softmax 归一化（按行）

**第 1 行** $[1.0,\; 0.0,\; 0.5]$：
$$\begin{aligned}
e^{1.0} &\approx 2.7183 \\
e^{0.0} &= 1.0000 \\
e^{0.5} &\approx 1.6487 \\
\sum &= 5.3670
\end{aligned}$$
$$\text{softmax}(\text{row}_1) = \left[\frac{2.7183}{5.3670},\; \frac{1.0000}{5.3670},\; \frac{1.6487}{5.3670}\right] = [0.5065,\; 0.1863,\; 0.3072]$$

**第 2 行** $[0.0,\; 1.0,\; 0.5]$（同理）：
$$\text{softmax}(\text{row}_2) = [0.1863,\; 0.5065,\; 0.3072]$$

$$A = \text{softmax}(S) = \begin{bmatrix} 0.5065 & 0.1863 & 0.3072 \\ 0.1863 & 0.5065 & 0.3072 \end{bmatrix}$$

### 步骤 3：加权求和 $O = A \times V$

**第 1 行输出**（查询 1 的结果）：
$$\begin{aligned}
o_1 &= 0.5065 \cdot [1,0,0,1,0] + 0.1863 \cdot [0,1,0,0,1] + 0.3072 \cdot [0,0,1,0,0] \\
&= [0.5065,\; 0.1863,\; 0.3072,\; 0.5065,\; 0.1863]
\end{aligned}$$

**第 2 行输出**（查询 2 的结果）：
$$\begin{aligned}
o_2 &= 0.1863 \cdot [1,0,0,1,0] + 0.5065 \cdot [0,1,0,0,1] + 0.3072 \cdot [0,0,1,0,0] \\
&= [0.1863,\; 0.5065,\; 0.3072,\; 0.1863,\; 0.5065]
\end{aligned}$$

### 最终输出矩阵

$$O = \text{Attention}(Q, K, V) = \begin{bmatrix} 0.5065 & 0.1863 & 0.3072 & 0.5065 & 0.1863 \\ 0.1863 & 0.5065 & 0.3072 & 0.1863 & 0.5065 \end{bmatrix} \in \mathbb{R}^{2 \times 5}$$

In [5]:
import numpy as np

def softmax(x, axis=-1):
    """Softmax 函数（数值稳定）"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, d_k):
    """缩放点积注意力"""
    # scores: (..., seq_len_q, seq_len_k)
    scores = np.dot(Q, K.swapaxes(-2, -1)) / np.sqrt(d_k)
    attn_weights = softmax(scores, axis=-1)
    output = np.dot(attn_weights, V)
    return output


def multi_head_attention_forward(X, num_heads=2, d_model=4):
    """
    多头注意力（Multi-Head Attention）前向传播

    参数:
        X: 输入 (seq_len, batch, d_model)
        num_heads: 注意力头数 (默认 2)
        d_model: 模型维度 (默认 4)

    返回:
        output: 多头注意力输出 (seq_len, batch, d_model)
    """
    seq_len, batch, d_model = X.shape
    d_k = d_model // num_heads  # 每个头的维度
    d_v = d_model // num_heads

    # 初始化投影权重（实际应用中为可学习参数）
    np.random.seed(42)
    W_q = np.random.randn(d_model, d_model) * 0.1
    W_k = np.random.randn(d_model, d_model) * 0.1
    W_v = np.random.randn(d_model, d_model) * 0.1
    W_o = np.random.randn(d_model, d_model) * 0.1

    # 1. 线性投影得到 Q, K, V (seq_len, batch, d_model)
    Q = np.dot(X, W_q)
    K = np.dot(X, W_k)
    V = np.dot(X, W_v)

    # 2. 拆分为多头: (seq_len, batch, d_model) -> (num_heads, seq_len, batch, d_k)
    def split_heads(x):
        s, b, _ = x.shape
        x = x.reshape(s, b, num_heads, d_k)       # (seq_len, batch, num_heads, d_k)
        x = x.transpose(2, 0, 1, 3)                # (num_heads, seq_len, batch, d_k)
        return x

    Q_heads = split_heads(Q)
    K_heads = split_heads(K)
    V_heads = split_heads(V)

    # 3. 对每个头计算缩放点积注意力
    head_outputs = []
    for h in range(num_heads):
        Q_h = Q_heads[h]  # (seq_len, batch, d_k)
        K_h = K_heads[h]
        V_h = V_heads[h]

        # 对 batch 中每个样本计算注意力
        head_out = np.zeros((seq_len, batch, d_v))
        for b_idx in range(batch):
            q = Q_h[:, b_idx, :]   # (seq_len, d_k)
            k = K_h[:, b_idx, :]   # (seq_len, d_k)
            v = V_h[:, b_idx, :]   # (seq_len, d_v)

            scores = np.dot(q, k.T) / np.sqrt(d_k)         # (seq_len, seq_len)
            attn_weights = softmax(scores, axis=1)          # (seq_len, seq_len)
            head_out[:, b_idx, :] = np.dot(attn_weights, v) # (seq_len, d_v)

        head_outputs.append(head_out)

    # 4. 拼接所有头: (num_heads, seq_len, batch, d_v) -> (seq_len, batch, d_model)
    concat = np.concatenate(head_outputs, axis=-1)  # (seq_len, batch, d_model)

    # 5. 最终线性投影
    output = np.dot(concat, W_o)  # (seq_len, batch, d_model)

    return output


# 测试
np.random.seed(42)
seq_len, batch, d_model = 3, 2, 4
X = np.random.randn(seq_len, batch, d_model)

print("输入 X shape:", X.shape)
print("X:\n", X)

output = multi_head_attention_forward(X, num_heads=2, d_model=4)
print("\n多头注意力输出 shape:", output.shape)
print("输出:\n", output)
print("\n输入输出形状一致:", X.shape == output.shape)

# 验证：输出不应为全零，且应有合理的数值范围
print(f"\n输出范围: [{output.min():.4f}, {output.max():.4f}]")
print(f"输出均值: {output.mean():.6f}")

输入 X shape: (3, 2, 4)
X:
 [[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
  [-0.23415337 -0.23413696  1.57921282  0.76743473]]

 [[-0.46947439  0.54256004 -0.46341769 -0.46572975]
  [ 0.24196227 -1.91328024 -1.72491783 -0.56228753]]

 [[-1.01283112  0.31424733 -0.90802408 -1.4123037 ]
  [ 1.46564877 -0.2257763   0.0675282  -1.42474819]]]

多头注意力输出 shape: (3, 2, 4)
输出:
 [[[ 0.00295132 -0.00066972 -0.00672659 -0.01064485]
  [-0.01534182 -0.00185725  0.03391203  0.0413105 ]]

 [[ 0.00311673 -0.00046337 -0.00668873 -0.01056854]
  [-0.015505   -0.00212144  0.03325691  0.04053742]]

 [[ 0.00317724 -0.00038515 -0.00667726 -0.01054224]
  [-0.01610522 -0.00238067  0.03250539  0.03986379]]]

输入输出形状一致: True

输出范围: [-0.0161, 0.0413]
输出均值: 0.005165
